# Set-Up

In [ ]:
# set up model
config_list = [
    {
        "model": "openai/gpt-4o-mini",
        "base_url": "https://openrouter.ai/api/v1",
        "api_key": ""
    }
]

In [ ]:
!pip -q install -U autogen-agentchat "autogen-ext[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 14.9 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

# input api key
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

Enter your OpenRouter API key: ··········


# Agents and Tools

In [ ]:
# import statements
import asyncio
import json
from datetime import datetime
import re
import matplotlib.pyplot as plt

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

# ======================================================
# MODEL CLIENT: OpenRouter
# ======================================================

model_client = OpenAIChatCompletionClient(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.4,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "structured_output": False,
        "family": "openai",
    },
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Spanish Multi-Agent Learning System"
    }
)

# ======================================================
# AGENTS
# ======================================================

# 1. Planner Agent - plans the current lesson
planner = AssistantAgent(
    name="planner",
    model_client=model_client,
    system_message=(
        "You are a Spanish Learning Planner. "
        "Only create short, practical lesson roadmaps for learning Spanish. "
        "Adapt to the learner's level: beginner, intermediate, or advanced. "
        "Focus on useful real-world Spanish."
        "Do NOT teach the lesson. "
        "Do NOT create a conversation scenario. "
        "Do NOT create quiz questions. "
        "Do NOT evaluate answers. "
        "Keep it under 6 bullet points."
    ),
)

# 2. Teacher Agent - teach a short language lesson based on user inputs
teacher = AssistantAgent(
    name="teacher",
    model_client=model_client,
    system_message=(
        "You are a Spanish Teacher. "
        "Teach one lesson at a time using simple explanations. "
        "Include Spanish vocabulary, pronunciation help, grammar notes, and examples. "
        "Always translate important Spanish phrases into English. "
        "Keep the lesson beginner-friendly unless another level is specified."
    ),
)

# 3. Culture Agent - Teach cultural facts relevant to the lesson
culture_coach = AssistantAgent(
    name="culture_coach",
    model_client=model_client,
    system_message=(
        "You are a Spanish Culture Coach. "
        "Teach short cultural notes connected to the lesson topic. "
        "Include customs, etiquette, regional differences, or real-world context. "
        "Keep it beginner-friendly and avoid stereotypes. "
        "Do NOT create quizzes or evaluate answers."
    ),
)

# 4. Conversation Agent - Create a conversation dialogue
conversation_coach = AssistantAgent(
    name="conversation_coach",
    model_client=model_client,
    system_message=(
        "You are a Spanish Conversation Coach. "
        "Create a short real-life Spanish practice activity, such as introducing yourself, "
        "ordering food, asking for directions, or talking about hobbies. "
        "Ask the learner to respond in Spanish if learner answers are provided. "
        "Keep activities practical and encouraging."
    ),
)

# 5. Quiz Agent - Short quiz to test the lesson
quizzer = AssistantAgent(
    name="quizzer",
    model_client=model_client,
    system_message=(
        "You are a Spanish Quiz Agent. "
        "Generate exactly 5 short questions based on the lesson. "
        "Include a mix of vocabulary, translation, and grammar. "
        "Make the questions match the learner's level."
    ),
)

# 6. Evaluator Agent - Evaluates the quiz answers
evaluator = AssistantAgent(
    name="evaluator",
    model_client=model_client,
    system_message=(
        "You are a Spanish Evaluator. "
        "Review learner answers and explain corrections clearly, "
        "and recommend the next lesson focus. "
        "Be supportive and specific. "
        "End your message with SESSION_COMPLETE."
    ),
)



# ======================================================
# TOOLS
# ======================================================

PROGRESS_FILE = "spanish_progress.json"

# save progress from current lesson to JSON file; learning history for analytics
def save_progress(level, goal, topic_focus, quiz_score, notes):
    """Save learner progress to a local JSON file."""
    record = {
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "level": level,
        "goal": goal,
        "topic_focus": topic_focus,
        "quiz_score": quiz_score,
        "notes": notes
    }

    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            data = json.load(f)
    else:
        data = []

    data.append(record)

    with open(PROGRESS_FILE, "w") as f:
        json.dump(data, f, indent=4)

    return "Progress saved successfully."


# load the progress from previous lessons
def load_progress():
    """Load previous Spanish learning progress."""
    if not os.path.exists(PROGRESS_FILE):
        return "No progress saved yet."

    with open(PROGRESS_FILE, "r") as f:
        data = json.load(f)

    return data


# analyze loaded progress
def analyze_progress(data):
    if not data:
        return "No data yet."

    scores = [int(entry["quiz_score"]) for entry in data]

    avg = sum(scores) / len(scores)

    return {
        "average_score": avg,
        "trend": "improving" if scores[-1] > scores[0] else "steady"
    }


# reset progress
def reset_progress():
    """Erase all saved progress with confirmation."""

    confirm = input("Are you sure you want to delete ALL progress? (yes/no): ")

    if confirm.lower() != "yes":
        return "Reset cancelled."

    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "w") as f:
            json.dump([], f, indent=4)
        return "All progress has been reset."

    return "No progress file found."



# Get the weeknesses from what needs more practice
def extract_weaknesses_from_evaluation(evaluation_text):
    """
    Extract weaknesses only from the section:
    ### What Needs More Practice:
    """

    pattern = r"### What Needs More Practice:\s*(.*?)(?=\n###|\Z)"

    match = re.search(pattern, evaluation_text, re.DOTALL | re.IGNORECASE)

    if not match:
        return []

    section_text = match.group(1).strip()

    weaknesses = []

    if "grammar" in section_text.lower():
        weaknesses.append("grammar")

    if "vocabulary" in section_text.lower():
        weaknesses.append("vocabulary")

    if "conversation" in section_text.lower() or "speaking" in section_text.lower():
        weaknesses.append("conversation")

    if "spelling" in section_text.lower() or "accent" in section_text.lower() or "accents" in section_text.lower():
        weaknesses.append("spelling/accents")

    if "pronunciation" in section_text.lower():
        weaknesses.append("pronunciation")

    if "complete sentence" in section_text.lower() or "sentence structure" in section_text.lower():
        weaknesses.append("sentence structure")

    return weaknesses

# recommend next lesson based on score
def recommend_next_lesson(score, topic_focus, goal, weaknesses=None):
    """
    Recommend next lesson based on performance and context.
    """

    score = int(score)
    weaknesses = weaknesses or []

    # Base recommendation from score
    if score >= 90:
        base = "advance"
    elif score >= 70:
        base = "review_and_expand"
    else:
        base = "repeat_foundations"

    # Topic-specific suggestions
    topic_map = {
        "introductions and greetings": [
            "asking questions (¿Cómo te llamas?, ¿De dónde eres?)",
            "talking about hobbies (Me gusta...)",
            "basic conversation flow"
        ],
        "ordering food": [
            "restaurant vocabulary",
            "polite phrases (Quisiera..., Por favor)",
            "asking for the bill"
        ],
        "travel": [
            "directions (¿Dónde está...?)",
            "transportation vocabulary",
            "booking hotels"
        ]
    }

    topic_suggestions = topic_map.get(topic_focus.lower(), ["expand vocabulary", "practice conversations"])

    # Weakness-based adjustments
    weakness_advice = []

    for w in weaknesses:
        if "grammar" in w.lower():
            weakness_advice.append("review basic sentence structure")
        if "vocabulary" in w.lower():
            weakness_advice.append("build vocabulary with flashcards")
        if "spelling" in w.lower() or "accent" in w.lower():
            weakness_advice.append("practice accents and spelling")
        if "conversation" in w.lower():
            weakness_advice.append("practice speaking with short dialogues")

    # Final recommendation logic
    if base == "advance":
        recommendation = f"Great progress! Move on to: {topic_suggestions[0]}"
    elif base == "review_and_expand":
        recommendation = f"Review {topic_focus}, then try: {topic_suggestions[1]}"
    else:
        recommendation = f"Repeat {topic_focus} and focus on fundamentals before advancing."

    # Add weakness-based advice
    if weakness_advice:
        recommendation += "\nFocus areas:\n- " + "\n- ".join(set(weakness_advice))

    # Align with goal
    recommendation += f"\nGoal alignment: Continue working toward '{goal}'."

    return recommendation


# suggest a topic focus based on level
def suggest_topic_focus(level):
    """Suggest Spanish lesson topics based only on level."""

    level = level.lower().strip()

    topics = {
        "beginner": [
            "introductions and greetings",
            "basic questions and answers",
            "numbers and dates",
            "days of the week and months",
            "telling time",
            "ordering food",
            "talking about hobbies",
            "family vocabulary",
            "describing people",
            "colors and clothing",
            "shopping basics",
            "asking for directions",
            "weather expressions",
            "common verbs (ser, estar, tener)",
            "simple present tense",
            "school vocabulary",
            "daily routines",
            "common phrases and expressions"
        ],
        "intermediate": [
            "past tense (preterite)",
            "imperfect tense usage",
            "travel situations",
            "hotel and airport conversations",
            "shopping and prices",
            "describing daily routines in detail",
            "making plans with friends",
            "giving and asking for opinions",
            "comparing things (más que, menos que)",
            "talking about the future",
            "using reflexive verbs",
            "health and doctor visits",
            "food preferences and cooking",
            "storytelling in Spanish",
            "describing experiences",
            "giving directions in detail",
            "cultural conversations"
        ],
        "advanced": [
            "subjunctive mood",
            "debating opinions",
            "workplace conversations",
            "formal vs informal speech",
            "news and current events",
            "advanced storytelling",
            "persuasive speaking",
            "expressing emotions and doubt",
            "hypothetical situations",
            "giving presentations",
            "writing formal emails",
            "cultural discussions",
            "idioms and slang",
            "analyzing texts",
            "argumentation in Spanish",
            "advanced grammar structures",
            "phrasal expressions"
        ]
    }

    return topics.get(level, topics["beginner"])


# Regional dialects
def get_region_settings(region):
    """Return Spanish style + pronunciation guidance based on region."""

    region = region.lower().strip()

    regions = {
        "mexico": {
            "name": "Mexico",
            "style": "Use Mexican Spanish vocabulary and examples.",
            "pronunciation": "Clear pronunciation. 's' sounds are preserved, and speech is generally slower and easier for learners."
        },
        "spain": {
            "name": "Spain",
            "style": "Use Spain Spanish vocabulary. Mention vosotros when useful.",
            "pronunciation": "Uses 'th' sound for 'z' and 'c' (like in 'gracias'). Distinction between 's' and 'z'."
        },
        "central america": {
            "name": "Central America",
            "style": "Use neutral Central American Spanish.",
            "pronunciation": "Generally clear pronunciation, though 's' may be softened in casual speech."
        },
        "caribbean": {
            "name": "Caribbean",
            "style": "Use Caribbean Spanish context (Puerto Rico, Cuba, Dominican Republic).",
            "pronunciation": "'s' is often dropped or softened. Faster speech and syllables may be shortened."
        },
        "south america": {
            "name": "South America",
            "style": "Use general South American Spanish.",
            "pronunciation": "Varies by country but generally clear, with some regional differences in speed and accent."
        },
        "argentina/uruguay": {
            "name": "Argentina/Uruguay",
            "style": "Use Rioplatense Spanish. Mention 'vos' instead of 'tú'.",
            "pronunciation": "'ll' and 'y' often sound like 'sh' or 'zh'. Distinct rhythm and intonation."
        },
        "andean": {
            "name": "Andean Region",
            "style": "Use Spanish from Peru, Bolivia, Ecuador.",
            "pronunciation": "Very clear and slower pronunciation. Good for learners."
        },
        "chile": {
            "name": "Chile",
            "style": "Use Chilean Spanish context.",
            "pronunciation": "Fast speech, dropped sounds, and unique slang can make it harder to understand."
        },
        "colombia/venezuela": {
            "name": "Colombia/Venezuela",
            "style": "Use Colombian/Venezuelan Spanish.",
            "pronunciation": "Very clear pronunciation, often considered among the easiest to understand."
        },
        "neutral": {
            "name": "Neutral Latin American Spanish",
            "style": "Use broadly understood Spanish without strong regional slang.",
            "pronunciation": "Clear, standard pronunciation suitable for learners."
        }
    }

    return regions.get(region, regions["neutral"])


# Regional slang
def get_region_slang(region):
    """Return common slang/expressions for a Spanish-speaking region."""

    region = region.lower().strip()

    slang = {
        "mexico": {
            "notes": "Use light Mexican slang when appropriate",
            "examples": [
                "¿Qué onda? = What's up?",
                "chido = cool",
                "mande = polite way to say 'what?' or 'pardon?'"
            ]
        },
        "spain": {
            "notes": "Use Spain-specific casual expressions when appropriate.",
            "examples": [
                "vale = okay",
                "guay = cool",
                "tío/tía = dude/person"
            ]
        },
        "central america": {
            "notes": "Use broad Central American expressions lightly.",
            "examples": [
                "¿Qué tal? = How's it going?",
                "pura vida = good vibes/life is good, especially Costa Rica",
                "chévere = cool, used in some areas"
            ]
        },
        "caribbean": {
            "notes": "Use Caribbean expressions lightly and explain them clearly.",
            "examples": [
                "¿Qué lo que? = What's up? common in Dominican Spanish",
                "chévere = cool",
                "guagua = bus in some Caribbean regions"
            ]
        },
        "south america": {
            "notes": "Use general South American expressions, but avoid assuming one country fits all.",
            "examples": [
                "chévere = cool, common in several countries",
                "bacán = cool, used in some countries",
                "plata = money, common in many regions"
            ]
        },
        "argentina/uruguay": {
            "notes": "Use Rioplatense expressions and explain vos forms when useful.",
            "examples": [
                "che = hey",
                "boludo/a = dude/friend, informal and context-dependent",
                "re = very, as in 're bueno'"
            ]
        },
        "andean": {
            "notes": "Use Andean-region expressions lightly and clearly.",
            "examples": [
                "chévere = cool",
                "bacán = cool",
                "ahorita = right now/soon, meaning varies by country"
            ]
        },
        "chile": {
            "notes": "Use Chilean slang carefully because it can be very region-specific.",
            "examples": [
                "bacán = cool",
                "po = emphasis particle",
                "¿cachai? = you know?/do you understand?"
            ]
        },
        "colombia/venezuela": {
            "notes": "Use Colombian/Venezuelan expressions lightly.",
            "examples": [
                "chévere = cool",
                "parce = friend/dude, common in Colombia",
                "pana = friend, common in Venezuela"
            ]
        },
        "neutral": {
            "notes": "Avoid strong slang. Use broadly understood Spanish.",
            "examples": [
                "¿Qué tal? = How's it going?",
                "muy bien = very good",
                "genial = great"
            ]
        }
    }

    return slang.get(region, slang["neutral"])


# region-aware conversation
def region_conversation_rules(region_settings, region_slang, level):
    level = level.lower()

    if level == "beginner":
        slang_rule = "Use little to no slang. Explain any regional phrase clearly."
    elif level == "intermediate":
        slang_rule = "Use a few regional slang expression naturally."
    else:
        slang_rule = "Use regional expressions naturally, but explain difficult ones."

    return f"""
        Region: {region_settings["name"]}
        Style: {region_settings["style"]}
        Pronunciation: {region_settings["pronunciation"]}
        Slang rule: {slang_rule}
        Regional slang examples:
        {chr(10).join("- " + item for item in region_slang["examples"])}
        """

# Lesson Instructions

In [ ]:
# ======================================================
# TEAM SETUP (OPTIONAL)
# ======================================================

termination = MaxMessageTermination(6)

# optional groupchat setup for non-interactive version
spanish_team = RoundRobinGroupChat(
    [planner, teacher, conversation_coach, quizzer, evaluator],
    termination_condition=termination,
)


# ======================================================
# RUN LESSON WITH USER ANSWERS
# ======================================================

async def run_full_interactive_spanish_session(
    level="beginner", # user input when run
    goal="have basic conversations in Spanish",
    topic_focus="introductions and greetings",
    region_settings=None,
    region_slang=None,
    conversation_turns=3,
    quiz_questions=5
):
    if region_settings is None:
        region_settings = get_region_settings("neutral")

    if region_slang is None:
        region_slang = get_region_slang("neutral")

    # 1. Planner
    plan_task = f"""
Create ONLY a short Spanish lesson roadmap.

Learner level: {level}
Goal: {goal}
Lesson focus: {topic_focus}

Keep it under 6 bullet points.
"""
    plan_result = await planner.run(task=plan_task)
    print("\n===== ROADMAP =====\n")
    print(plan_result.messages[-1].content)
    print("\n===== PREVIOUS PROGRESS =====\n")
    print(load_progress())

    # 2. Teacher
    lesson_task = f"""
Teach one compact Spanish lesson.

Learner level: {level}
Goal: {goal}
Lesson focus: {topic_focus}

Region focus: {region_settings["name"]}
Spanish style: {region_settings["style"]}
Pronunciation note: {region_settings["pronunciation"]}

Regional slang guidance: {region_slang["notes"]}
Useful regional expressions:
{chr(10).join("- " + item for item in region_slang["examples"])}

Include:
- key vocabulary
- pronunciation help
- grammar notes
- examples with English translations
- 1-2 regional slang examples if appropriate

Do NOT create a quiz.
Do NOT evaluate.
"""
    lesson_result = await teacher.run(task=lesson_task)
    lesson_text = lesson_result.messages[-1].content
    print("\n===== REGION SETTINGS =====")
    print("Region:", region_settings["name"])
    print("Style:", region_settings["style"])
    print("Pronunciation:", region_settings["pronunciation"])
    print("\n===== LESSON =====\n")
    print(lesson_text)

# Culture section
    culture_task = f"""
Create a short Spanish culture section related to this lesson.

Learner level: {level}
Lesson focus: {topic_focus}

Region focus: {region_settings["name"]}
Spanish style: {region_settings["style"]}

Include:
- one cultural note
- one useful etiquette tip
- one regional variation based on the selected region if relevant
- keep it short

Do NOT quiz.
Do NOT evaluate.
"""

    culture_result = await culture_coach.run(task=culture_task)
    culture_text = culture_result.messages[-1].content
    print("\n===== CULTURE NOTE =====\n")
    print(culture_text)

    # 3. Conversation setup
    regional_rules = region_conversation_rules(region_settings, region_slang, level)
    convo_setup_task = f"""
Create a short Spanish conversation practice based on this lesson:

{lesson_text}

Region focus: {region_settings["name"]}
Spanish style: {region_settings["style"]}
Pronunciation note: {region_settings["pronunciation"]}
Regional conversation rules:{regional_rules}

Regional slang guidance: {region_slang["notes"]}
Useful regional expressions:
{chr(10).join("- " + item for item in region_slang["examples"])}

Do:
- Give a 1-2 sentence scenario in English
- Start the conversation with ONE Spanish line from the other person
- 1-2 regional slang examples if appropriate
"""
    convo_setup = await conversation_coach.run(task=convo_setup_task)
    print("\n===== CONVERSATION PRACTICE =====\n")
    print(convo_setup.messages[-1].content)

    # 4. Live conversation
    conversation_history = []

    for i in range(conversation_turns):
        user_reply = input(f"\nConversation reply {i+1} in Spanish: ")
        conversation_history.append(f"Student: {user_reply}")

        convo_reply_task = f"""
Continue this beginner Spanish conversation.
Regional conversation rules: {regional_rules}

Conversation so far:
{chr(10).join(conversation_history)}

Rules:
- Reply as the other person in Spanish in 1-2 simple sentences.
- Then give brief feedback in English.
- Correct grammar, spelling, and accents if needed.
- Give one improved version of the student's sentence.
"""
        convo_reply = await conversation_coach.run(task=convo_reply_task)
        coach_text = convo_reply.messages[-1].content

        print("\nCoach:\n")
        print(coach_text)

        conversation_history.append(f"Coach: {coach_text}")

    # 5. Quiz
    quiz_task = f"""
Create exactly {quiz_questions} numbered quiz questions based on this Spanish lesson:

{lesson_text}

Region focus: {region_settings["name"]}
Spanish style: {region_settings["style"]}
Pronunciation note: {region_settings["pronunciation"]}

Regional slang guidance: {region_slang["notes"]}
Useful regional expressions:
{chr(10).join("- " + item for item in region_slang["examples"])}

Include a mix of:
- vocabulary
- translation
- grammar
- short sentence creation
- 1-2 regional slang examples if appropriate

Do NOT include answers.
"""
    quiz_result = await quizzer.run(task=quiz_task)
    quiz_text = quiz_result.messages[-1].content

    print("\n===== QUIZ =====\n")
    print(quiz_text)

    # 6. Live quiz answers
    print("\n===== YOUR QUIZ ANSWERS =====\n")

    quiz_answers = []
    for i in range(1, quiz_questions + 1):
        answer = input(f"Answer question {i}: ")
        quiz_answers.append(f"{i}. {answer}")

    quiz_answers_text = "\n".join(quiz_answers)

    # 7. Evaluator
    evaluation_task = f"""
Evaluate this Spanish learning session.

Learner level: {level}
Goal: {goal}
Lesson focus: {topic_focus}

Region focus: {region_settings["name"]}
Spanish style: {region_settings["style"]}
Pronunciation note: {region_settings["pronunciation"]}

Lesson:
{lesson_text}

Culture section:
{culture_text}

Conversation history:
{chr(10).join(conversation_history)}

Quiz:
{quiz_text}

Learner quiz answers:
{quiz_answers_text}

Provide:
- quiz score
- conversation feedback
- corrections
- what was done well
- what needs more practice
- what to study next

End with SESSION_COMPLETE.
"""
    evaluation_result = await evaluator.run(task=evaluation_task)

    print("\n===== FINAL EVALUATION =====\n")
    print(evaluation_result.messages[-1].content)

    evaluation_text = evaluation_result.messages[-1].content
    weaknesses = extract_weaknesses_from_evaluation(evaluation_text)

    # ======================================================
    # TOOL USE: Save progress + recommend next lesson
    # ======================================================

    score_input = input("\nEnter the quiz score from the evaluator: ")
    score_input = int(score_input)

    next_lesson = recommend_next_lesson(
        score=score_input,
        topic_focus=topic_focus,
        goal=goal,
        weaknesses=weaknesses
    )

    save_message = save_progress(
        level=level,
        goal=goal,
        topic_focus=topic_focus,
        quiz_score=score_input,
        notes=next_lesson
    )

    # Load all past progress
    progress_data = load_progress()
    # Analyze it
    analytics = analyze_progress(progress_data)
    # Display
    print("\n===== PROGRESS ANALYTICS =====\n")
    if isinstance(analytics, dict):
        print(f"Average Score: {analytics['average_score']:.2f}")
        print(f"Trend: {analytics['trend']}")
    else:
        print(analytics)

    print("\n===== TOOL RESULTS =====\n")
    print(save_message)
    print("Recommended next lesson:", next_lesson)


    return {
        "plan": plan_result,
        "lesson": lesson_result,
        "culture": culture_result,
        "conversation_history": conversation_history,
        "quiz": quiz_result,
        "quiz_answers": quiz_answers_text,
        "evaluation": evaluation_result,
    }

# Main Menu

In [ ]:
async def main_menu():
    while True:
        print("\n===== SPANISH LEARNING SYSTEM MENU =====")
        print("1. Start new lesson")
        print("2. View progress")
        print("3. View progress analytics")
        print("4. Reset progress")
        print("5. Exit")

        choice = input("Choose an option: ")

        if choice == "1":
            level = input("Choose level (beginner/intermediate/advanced): ")
            goal = input("What is your goal? ")

            suggestions = suggest_topic_focus(level)
            print("\n===== SUGGESTED TOPICS =====")
            for i, topic in enumerate(suggestions, start=1):
                print(f"{i}. {topic}")
            topic_choice = input("Choose a topic number or type your own topic: ")
            if topic_choice.isdigit() and 1 <= int(topic_choice) <= len(suggestions):
                topic_focus = suggestions[int(topic_choice) - 1]
            else:
                topic_focus = topic_choice or suggestions[0]

            # 3. REGION SELECTION
            print("\nChoose a Spanish region/style:")
            print("1. Mexico")
            print("2. Spain")
            print("3. Central America")
            print("4. Caribbean")
            print("5. South America")
            print("6. Argentina/Uruguay")
            print("7. Andean Region")
            print("8. Chile")
            print("9. Colombia/Venezuela")
            print("10. Neutral Latin American Spanish")

            region_choice = input("Choose a region number: ")

            region_options = {
                "1": "mexico",
                "2": "spain",
                "3": "central america",
                "4": "caribbean",
                "5": "south america",
                "6": "argentina/uruguay",
                "7": "andean",
                "8": "chile",
                "9": "colombia/venezuela",
                "10": "neutral"
            }

            region = region_options.get(region_choice, "neutral")
            region_settings = get_region_settings(region)
            region_slang = get_region_slang(region)

            await run_full_interactive_spanish_session(
                level=level,
                goal=goal,
                topic_focus=topic_focus,
                region_settings=region_settings,
                region_slang=region_slang
            )

        elif choice == "2":
            print("\n===== SAVED PROGRESS =====")
            progress = load_progress()
            print(progress)

        elif choice == "3":
            print("\n===== PROGRESS ANALYTICS =====")
            progress = load_progress()
            analytics = analyze_progress(progress)
            print(analytics)

        elif choice == "4":
            print(reset_progress())

        elif choice == "5":
            print("Goodbye! ¡Hasta luego!")
            break

        else:
            print("Invalid choice. Please enter 1, 2, 3, 4, or 5.")

# Run System

In [ ]:
await main_menu()


===== SPANISH LEARNING SYSTEM MENU =====
1. Start new lesson
2. View progress
3. View progress analytics
4. Reset progress
5. Exit
Choose an option: 1
Choose level (beginner/intermediate/advanced): intermediate
What is your goal? write complex sentences

===== SUGGESTED TOPICS =====
1. past tense (preterite)
2. imperfect tense usage
3. travel situations
4. hotel and airport conversations
5. shopping and prices
6. describing daily routines in detail
7. making plans with friends
8. giving and asking for opinions
9. comparing things (más que, menos que)
10. talking about the future
11. using reflexive verbs
12. health and doctor visits
13. food preferences and cooking
14. storytelling in Spanish
15. describing experiences
16. giving directions in detail
17. cultural conversations
Choose a topic number or type your own topic: 2

Choose a Spanish region/style:
1. Mexico
2. Spain
3. Central America
4. Caribbean
5. South America
6. Argentina/Uruguay
7. Andean Region
8. Chile
9. Colombia/Vene